In [ ]:
from pathlib import Path
import json

# Find the root directory, no matter where we are. 
def find_project_root(marker="README.md"):
    p = Path.cwd()
    while p != p.parent:
        if (p / marker).exists():
            return p
        p = p.parent
    raise RuntimeError("Project root not found")

PROJECT_ROOT = find_project_root()

path = PROJECT_ROOT / "notebooks/api_progress.json"

In [ ]:
# Read in feature engineered data 

input_path = PROJECT_ROOT / "data/processed/feature_engineered_data.csv"
import pandas as pd
df = pd.read_csv(input_path)

In [ ]:
input_path = PROJECT_ROOT / "data/progress_tracking/urls_with_descriptions.csv"
descriptions = pd.read_csv(input_path)

In [ ]:
descriptions.rename(columns={'description': 'full_description'}, inplace=True)

In [ ]:
df = df.merge(descriptions[['redirect_url', 'full_description']], on='redirect_url', how='left')

In [ ]:
df['full_description']  = df['full_description'].fillna('')
df['full_description_length'] = df['full_description'].str.len()

In [ ]:
X_structured = df.drop(columns=['avg_salary', 'title', 'description'])
X_structured.columns

cat_vars = ['contract_type', 'contract_time', 'category.label', 'location.area_length', 
            # 'location_country', 
            'location_state', 'location_region_abridged', 'location_city_abridged', 'missing_long_lat']

num_vars = ['longitude', 'latitude']

In [ ]:
# Model training and validation functions 

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sentence_transformers import SentenceTransformer
import pandas as pd

EMBEDDING_MODEL = SentenceTransformer('all-MiniLM-L6-v2')


def get_text_series(df, text_config):
    """
    Select the appropriate text column from the dataframe.
    text_config: 'title', 'description', or 'title_description'
    """
    valid = ['title', 'description', 'title_description', 'full_description']
    if text_config not in valid:
        raise ValueError(f"text_config must be one of {valid}, got '{text_config}'")
    return df[text_config]


def build_text_features(train_series, val_series, feature_config, reduce=False, n_components=100):
    """
    Fit text transformers on train_series, transform both train and val.
    feature_config: 'tfidf', 'embeddings', or 'both'
    Returns X_train, X_val as numpy arrays.
    """

    def get_tfidf(train, val):
        tfidf = TfidfVectorizer(max_features=10_000, ngram_range=(1, 2), min_df=3, sublinear_tf=True)
        X_train = tfidf.fit_transform(train)
        X_val = tfidf.transform(val)
        if reduce:
            svd = TruncatedSVD(n_components=n_components, random_state=42)
            X_train = svd.fit_transform(X_train)
            X_val = svd.transform(X_val)
        else:
            X_train = X_train.toarray()
            X_val = X_val.toarray()
        return X_train, X_val

    def get_embeddings(train, val):
        X_train = EMBEDDING_MODEL.encode(train.tolist(), batch_size=64, show_progress_bar=False)
        X_val = EMBEDDING_MODEL.encode(val.tolist(), batch_size=64, show_progress_bar=False)
        if reduce:
            pca = PCA(n_components=n_components, random_state=42)
            X_train = pca.fit_transform(X_train)
            X_val = pca.transform(X_val)
        return X_train, X_val

    if feature_config == 'tfidf':
        return get_tfidf(train_series, val_series)
    elif feature_config == 'embeddings':
        return get_embeddings(train_series, val_series)
    elif feature_config == 'both':
        tfidf_train, tfidf_val = get_tfidf(train_series, val_series)
        emb_train, emb_val = get_embeddings(train_series, val_series)
        return np.hstack([tfidf_train, emb_train]), np.hstack([tfidf_val, emb_val])
    else:
        raise ValueError(f"feature_config must be 'tfidf', 'embeddings', or 'both', got '{feature_config}'")


def evaluate(y_true, y_pred):
    """
    Calculate regression metrics.
    Returns dict with rmse, mae, r2.
    """
    return {
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'mae': mean_absolute_error(y_true, y_pred),
        'r2': r2_score(y_true, y_pred)
    }

In [ ]:
# Reduce down to descriptions longer than 30 characters - we do this for both the non description and description model to use the same dataset for fairness. 
df = df[df['full_description_length'] > 30]

In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import numpy as np
from sklearn.impute import SimpleImputer

df = df.reset_index(drop=True)
y = df['avg_salary'].values

# encode categoricals
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
cat_encoded = ohe.fit_transform(df[cat_vars])


imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()

# scale continuous
cont_imputed = imputer.fit_transform(df[num_vars])
cont_scaled = scaler.fit_transform(cont_imputed)


# combine into one non-text feature matrix
X_structured = np.hstack([cat_encoded, cont_scaled])

In [ ]:
# Text the performance of a mdoel that reduces description but not title: 
n_splits = 5
use = 'text'
model_type = 'ridge'

kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
results = []


fold_metrics = []

for fold_num, (train_idx, val_idx) in enumerate(kf.split(df)):
    # Print statement so we can track progress more granularly
    print(f"  Fold {fold_num + 1}/{n_splits}")
    # slice text
    title_series = get_text_series(df, 'title')
    description_series = get_text_series(df, 'description')
    
    train_title_text = title_series.iloc[train_idx]
    val_title_text = title_series.iloc[val_idx]
    train_description_text = description_series.iloc[train_idx]
    val_description_text = description_series.iloc[val_idx]

    # build text features
    X_title_train, X_title_val = build_text_features(
        train_title_text, val_title_text,
        feature_config='tfidf',
        reduce= False 
    )
    
    X_description_train, X_description_val = build_text_features(
        train_description_text, val_description_text,
        feature_config='tfidf',
        reduce= True
    )

    # slice and combine with non-text features
    
    if use == 'text':
        X_train = np.hstack([X_title_train, X_description_train])
        X_val = np.hstack([X_title_val, X_description_val])
    elif use == 'structured':
        X_train = X_structured[train_idx]
        X_val = X_structured[val_idx]
    elif use == 'both':
        X_train = np.hstack([X_title_train, X_description_train, X_structured[train_idx]])
        X_val = np.hstack([X_title_val, X_description_val, X_structured[val_idx]])
    else:
        raise ValueError(f"use must be 'text', 'structured', or 'both', got '{use}'")

    y_train = y[train_idx]
    y_val = y[val_idx]

    # fit model
    model = Ridge(alpha=1.0) if model_type == 'ridge' else Lasso(alpha=0.1)
    model.fit(X_train, y_train)

    # evaluate
    y_pred = model.predict(X_val)
    fold_metrics.append(evaluate(y_val, y_pred))

print(fold_metrics)

In [ ]:
# Test how much the full description adds to predictiive power. 
n_splits = 5
use = 'text'
model_type = 'ridge'

kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
results = []


fold_metrics = []

for fold_num, (train_idx, val_idx) in enumerate(kf.split(df)):
    # Print statement so we can track progress more granularly
    print(f"  Fold {fold_num + 1}/{n_splits}")
    # slice text
    title_series = get_text_series(df, 'title')
    # This is the only line that changes.
    description_series = get_text_series(df, 'full_description')
    
    train_title_text = title_series.iloc[train_idx]
    val_title_text = title_series.iloc[val_idx]
    train_description_text = description_series.iloc[train_idx]
    val_description_text = description_series.iloc[val_idx]

    # build text features
    X_title_train, X_title_val = build_text_features(
        train_title_text, val_title_text,
        feature_config='tfidf',
        reduce= False 
    )
    
    X_description_train, X_description_val = build_text_features(
        train_description_text, val_description_text,
        feature_config='tfidf',
        reduce= True
    )

    # slice and combine with non-text features
    
    if use == 'text':
        X_train = np.hstack([X_title_train, X_description_train])
        X_val = np.hstack([X_title_val, X_description_val])
    elif use == 'structured':
        X_train = X_structured[train_idx]
        X_val = X_structured[val_idx]
    elif use == 'both':
        X_train = np.hstack([X_title_train, X_description_train, X_structured[train_idx]])
        X_val = np.hstack([X_title_val, X_description_val, X_structured[val_idx]])
    else:
        raise ValueError(f"use must be 'text', 'structured', or 'both', got '{use}'")

    y_train = y[train_idx]
    y_val = y[val_idx]

    # fit model
    model = Ridge(alpha=1.0) if model_type == 'ridge' else Lasso(alpha=0.1)
    model.fit(X_train, y_train)

    # evaluate
    y_pred = model.predict(X_val)
    fold_metrics.append(evaluate(y_val, y_pred))

print(fold_metrics)

In [ ]:
n_splits = 5
use = 'structured'
model_type = 'ridge'

kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
results = []


fold_metrics = []

for fold_num, (train_idx, val_idx) in enumerate(kf.split(df)):
    # Print statement so we can track progress more granularly
    print(f"  Fold {fold_num + 1}/{n_splits}")
    # slice text
    title_series = get_text_series(df, 'title')
    # This is the only line that changes.
    description_series = get_text_series(df, 'full_description')
    
    train_title_text = title_series.iloc[train_idx]
    val_title_text = title_series.iloc[val_idx]
    train_description_text = description_series.iloc[train_idx]
    val_description_text = description_series.iloc[val_idx]

    # slice and combine with non-text features
    
    if use == 'text':
        X_train = np.hstack([X_title_train, X_description_train])
        X_val = np.hstack([X_title_val, X_description_val])
    elif use == 'structured':
        X_train = X_structured[train_idx]
        X_val = X_structured[val_idx]
    elif use == 'both':
        X_train = np.hstack([X_title_train, X_description_train, X_structured[train_idx]])
        X_val = np.hstack([X_title_val, X_description_val, X_structured[val_idx]])
    else:
        raise ValueError(f"use must be 'text', 'structured', or 'both', got '{use}'")

    y_train = y[train_idx]
    y_val = y[val_idx]

    # fit model
    model = Ridge(alpha=1.0) if model_type == 'ridge' else Lasso(alpha=0.1)
    model.fit(X_train, y_train)

    # evaluate
    y_pred = model.predict(X_val)
    fold_metrics.append(evaluate(y_val, y_pred))

print(fold_metrics)

In [ ]:
# Text the performance of a mdoel that reduces description but not title (and includes structured data): 
n_splits = 5
use = 'both'
model_type = 'ridge'

kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
results = []


fold_metrics = []

for fold_num, (train_idx, val_idx) in enumerate(kf.split(df)):
    # Print statement so we can track progress more granularly
    print(f"  Fold {fold_num + 1}/{n_splits}")
    # slice text
    title_series = get_text_series(df, 'title')
    description_series = get_text_series(df, 'description')
    
    train_title_text = title_series.iloc[train_idx]
    val_title_text = title_series.iloc[val_idx]
    train_description_text = description_series.iloc[train_idx]
    val_description_text = description_series.iloc[val_idx]

    # build text features
    X_title_train, X_title_val = build_text_features(
        train_title_text, val_title_text,
        feature_config='tfidf',
        reduce= False 
    )
    
    X_description_train, X_description_val = build_text_features(
        train_description_text, val_description_text,
        feature_config='tfidf',
        reduce= True
    )

    # slice and combine with non-text features
    
    if use == 'text':
        X_train = np.hstack([X_title_train, X_description_train])
        X_val = np.hstack([X_title_val, X_description_val])
    elif use == 'structured':
        X_train = X_structured[train_idx]
        X_val = X_structured[val_idx]
    elif use == 'both':
        X_train = np.hstack([X_title_train, X_description_train, X_structured[train_idx]])
        X_val = np.hstack([X_title_val, X_description_val, X_structured[val_idx]])
    else:
        raise ValueError(f"use must be 'text', 'structured', or 'both', got '{use}'")

    y_train = y[train_idx]
    y_val = y[val_idx]

    # fit model
    model = Ridge(alpha=1.0) if model_type == 'ridge' else Lasso(alpha=0.1)
    model.fit(X_train, y_train)

    # evaluate
    y_pred = model.predict(X_val)
    fold_metrics.append(evaluate(y_val, y_pred))

print(fold_metrics)

In [ ]:
# Test how much the full description adds to predictiive power (with structured data). 
n_splits = 5
use = 'both'
model_type = 'ridge'

kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
results = []


fold_metrics = []

for fold_num, (train_idx, val_idx) in enumerate(kf.split(df)):
    # Print statement so we can track progress more granularly
    print(f"  Fold {fold_num + 1}/{n_splits}")
    # slice text
    title_series = get_text_series(df, 'title')
    # This is the only line that changes.
    description_series = get_text_series(df, 'full_description')
    
    train_title_text = title_series.iloc[train_idx]
    val_title_text = title_series.iloc[val_idx]
    train_description_text = description_series.iloc[train_idx]
    val_description_text = description_series.iloc[val_idx]

    # build text features
    X_title_train, X_title_val = build_text_features(
        train_title_text, val_title_text,
        feature_config='tfidf',
        reduce= False 
    )
    
    X_description_train, X_description_val = build_text_features(
        train_description_text, val_description_text,
        feature_config='tfidf',
        reduce= True
    )

    # slice and combine with non-text features
    
    if use == 'text':
        X_train = np.hstack([X_title_train, X_description_train])
        X_val = np.hstack([X_title_val, X_description_val])
    elif use == 'structured':
        X_train = X_structured[train_idx]
        X_val = X_structured[val_idx]
    elif use == 'both':
        X_train = np.hstack([X_title_train, X_description_train, X_structured[train_idx]])
        X_val = np.hstack([X_title_val, X_description_val, X_structured[val_idx]])
    else:
        raise ValueError(f"use must be 'text', 'structured', or 'both', got '{use}'")

    y_train = y[train_idx]
    y_val = y[val_idx]

    # fit model
    model = Ridge(alpha=1.0) if model_type == 'ridge' else Lasso(alpha=0.1)
    model.fit(X_train, y_train)

    # evaluate
    y_pred = model.predict(X_val)
    fold_metrics.append(evaluate(y_val, y_pred))

print(fold_metrics)

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
# Try a gradient boosting model to see if we have any non-lineaities of interest. 
gb = GradientBoostingRegressor(n_estimators=200, max_depth=4, random_state=42)


# Test how much the full description adds to predictiive power (with structured data). 
n_splits = 5
use = 'both'
model_type = 'gb'

kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
results = []


fold_metrics = []

for fold_num, (train_idx, val_idx) in enumerate(kf.split(df)):
    # Print statement so we can track progress more granularly
    print(f"  Fold {fold_num + 1}/{n_splits}")
    # slice text
    title_series = get_text_series(df, 'title')
    # This is the only line that changes.
    description_series = get_text_series(df, 'full_description')
    
    train_title_text = title_series.iloc[train_idx]
    val_title_text = title_series.iloc[val_idx]
    train_description_text = description_series.iloc[train_idx]
    val_description_text = description_series.iloc[val_idx]

    # build text features
    X_title_train, X_title_val = build_text_features(
        train_title_text, val_title_text,
        feature_config='tfidf',
        reduce= False 
    )
    
    X_description_train, X_description_val = build_text_features(
        train_description_text, val_description_text,
        feature_config='tfidf',
        reduce= True
    )

    # slice and combine with non-text features
    
    if use == 'text':
        X_train = np.hstack([X_title_train, X_description_train])
        X_val = np.hstack([X_title_val, X_description_val])
    elif use == 'structured':
        X_train = X_structured[train_idx]
        X_val = X_structured[val_idx]
    elif use == 'both':
        X_train = np.hstack([X_title_train, X_description_train, X_structured[train_idx]])
        X_val = np.hstack([X_title_val, X_description_val, X_structured[val_idx]])
    else:
        raise ValueError(f"use must be 'text', 'structured', or 'both', got '{use}'")

    y_train = y[train_idx]
    y_val = y[val_idx]

    # fit model
    model = gb if model_type == 'gb' else Ridge(alpha=0.1)
    model.fit(X_train, y_train)

    # evaluate
    y_pred = model.predict(X_val)
    fold_metrics.append(evaluate(y_val, y_pred))

print(fold_metrics)